# Product Scraping
> 실제 상품 페이지에서 정보를 수집하고 저장하는 과정을 연습합니다.

2026년 데이터 파이프라인 학습 내용을 정리한 실습 노트입니다.

## 데이터 수집 자동화

### 다나와 상품정보 스크래핑

#### 자동 로그인

- 패스워드 숨기기 

##### 1. setx 명령어로 시스템에 저장방법
- powershell

```powershell
> setx DANAWA_ID "아이디"
성공: 지정한 값을 저장했습니다.

> setx DANAWA_PASSWORD "비밀번호"

성공: 지정한 값을 저장했습니다.
```

- sysdm.cpl로 시스템 정보에서 환경변수 user 리스트에서 오픈

##### 2. keyring 으로 윈도우 자격증명 저장소

- setx 와 달리 아예 오픈 안됨
- 패키지 설치

In [ ]:
import keyring
from getpass import getpass

In [ ]:
username = input('다나와 아이디')
password = getpass('다나와 비밀번호') 

# 아이디 저장
keyring.set_password('danawa.com', 'user_id', username)
# 패스워드
keyring.set_password('danawa.com', username, password)

print('아이디/패스워드 저장 완료')

In [ ]:
user_id = keyring.get_password('danawa.com', 'user_id')
user_id

In [ ]:
password = keyring.get_password('danawa.com', user_id)
password is not None

- 패스워드 오픈 

In [ ]:
import pandas as pd
import os

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [ ]:
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 10)
driver.get('https://www.danawa.com/')

wait.until(
    EC.presence_of_element_located((By.ID, 'danawa_footer'))
)

In [ ]:
# 로그인 링크 확인후 클릭
driver.find_element(By.LINK_TEXT, '로그인').click()

In [ ]:
# 로그인하기
id_input = driver.find_element(By.CSS_SELECTOR, 'input[type="text"]')
id_input.send_keys(user_id)

pass_input = driver.find_element(By.CSS_SELECTOR, 'input[type="password"]')
pass_input.send_keys(password)

In [ ]:
login_btn = driver.find_element(By.CLASS_NAME, 'btn_login')
login_btn.click()

#### 동적 페이지 스크래핑

- https://prod.danawa.com/list/?cate=11255834&15main_11_02

In [ ]:
driver.get('https://prod.danawa.com/list/?cate=11255834&15main_11_02')

# wait.until(
#     EC.presence_of_element_located((By.TAG_NAME, 'footer'))
# )
wait.until(
    EC.visibility_of_element_located((By.TAG_NAME, 'footer'))
)

- select 태그 페이지 내에 1개만 존재

In [ ]:
options = driver.find_elements(By.TAG_NAME, 'option')

for option in options:
    if option.get_attribute('value') == '90':
        option.click()

wait.until(
    EC.visibility_of_element_located((By.TAG_NAME, 'footer'))
)

In [ ]:
products = driver.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListItem"]')
len(products)

In [ ]:
product = products[1]
product.text

In [ ]:
## 1번 인덱스 값에서 제품명, 제품스펙, 2번 인덱스 값에서 가격 
product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')


In [ ]:
computer_info = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')[1]
computer_info.text

In [ ]:
# 컴퓨터 이름 추출!!
computer_info.find_element(By.CSS_SELECTOR, 'div a').text

In [ ]:
# 컴퓨터 사양
specs = computer_info.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListSpecs"]')
' / '.join(spec.text for spec in specs)

In [ ]:
price_info = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')[2]
price_info.text

In [ ]:
# 가격정보 추출
price_info.find_element(By.CSS_SELECTOR, 'li').text

In [ ]:
# 인덱스로 데이터 가져오면서 필요값만 필터링
# 원이나오면 반복 종료
price_info.find_elements(By.CSS_SELECTOR, 'li > div span')[0].text

In [ ]:
spans = price_info.find_elements(By.CSS_SELECTOR, 'li > div span') 

for span in spans:
    if ('위' in span.text): continue
    if ('원' in span.text): break    
    print(span.text)

- 각 부분별 동작확인 완료

##### 한페이지 90개 데이터 스크래핑

In [ ]:
# 다나와 게시판 게이밍PC 페이지로 이동
driver.get('https://prod.danawa.com/list/?cate=11255834&15main_11_02')

wait.until(
    # 모든 요소가 화면에 완전표시될때까지 대기
    EC.visibility_of_all_elements_located((By.TAG_NAME, 'footer'))
)

# 게시물 개수 30개에서 90개로 늘리기
options = driver.find_elements(By.TAG_NAME, 'option')

for option in options:
    if option.get_attribute('value') == '90':
        option.click()

# 화면에 90개 데이터를 다 조회할때까지 대기
products = wait.until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div[data-testid="ProductListItem"]'))
)

# list를 담을 수 있는 배열 생성
danawa_prices = []

# 총 게시물 읽어오기
# len(products)
for index, product in enumerate(products):
    # print(index, product.text)
    items = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')
    computer_info = items[1]  # 컴퓨터 명, 사양 정보 들어있는 영역
    price_info = items[2]   # 가격정보 들어있는 영역

    # print(computer_info.text)
    ## Pandas DF에 컬럼으로 사용할 이름들 사용
    ## index, computer_name, computer_spec
    computer_name = computer_info.find_element(By.CSS_SELECTOR, 'div a').text
    specs = computer_info.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListSpecs"]')
    computer_spec = ' / '.join(spec.text for spec in specs)

    # 가격
    spans = price_info.find_elements(By.CSS_SELECTOR, 'li > div span') 
    price_spec = ''
    price = ''

    for span in spans:
        if ('위' in span.text): continue  # 순위 제외
        if ('원' in span.text): break    # 첫번째 금액만 받고 종료

        if ('B' in span.text):
            price_spec = span.text
        else:
            price = span.text

    danawa_prices.append({
        'index': index,
        'computer_name': computer_name,
        'computer_spec': computer_spec,
        'price_spec' : price_spec,
        'price' : price
    })       
    

In [ ]:
len( danawa_prices)

In [ ]:
## Pandas DF로 변환
df_danawa = pd.DataFrame(danawa_prices)

In [ ]:
df_danawa.tail()

##### 페이지 확인


In [ ]:
# 
buttons = driver.find_elements(By.CSS_SELECTOR, 'nav > nav div button')
len(buttons)

In [ ]:
for button in buttons:
    print(int(button.text))

In [ ]:
button_nextpage = buttons[9]
button_nextpage.click()

In [ ]:
# 다 동작했을때 10페이지에 있는지 확인
for button in buttons:
    button.click()

    products = wait.until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div[data-testid="ProductListItem"]'))
    )


##### 이전 스크래핑 + 페이지 이동 조합
- 통합 소스코드

In [ ]:
# 다나와 게시판 게이밍PC 페이지로 이동
driver.get('https://prod.danawa.com/list/?cate=11255834&15main_11_02')

wait.until(
    # 모든 요소가 화면에 완전표시될때까지 대기
    EC.visibility_of_all_elements_located((By.TAG_NAME, 'footer'))
)

# 게시물 개수 30개에서 90개로 늘리기
options = driver.find_elements(By.TAG_NAME, 'option')

for option in options:
    if option.get_attribute('value') == '90':
        option.click()

# list를 담을 수 있는 배열 생성
danawa_prices = []
buttons = driver.find_elements(By.CSS_SELECTOR, 'nav > nav div button')

# 10개 페이지 이동하면서 스크래핑
for button in buttons:
    button.click()
    page_num = int(button.text)

    products = wait.until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div[data-testid="ProductListItem"]'))
    )

    # 한페이지 스크래핑 내용이 여기 
    # 총 게시물 읽어오기
    # len(products)
    for index, product in enumerate(products):
        # print(index, product.text)
        items = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')
        computer_info = items[1]  # 컴퓨터 명, 사양 정보 들어있는 영역
        price_info = items[2]   # 가격정보 들어있는 영역

        # print(computer_info.text)
        ## Pandas DF에 컬럼으로 사용할 이름들 사용
        ## index, computer_name, computer_spec
        computer_name = computer_info.find_element(By.CSS_SELECTOR, 'div a').text
        specs = computer_info.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListSpecs"]')
        computer_spec = ' / '.join(spec.text for spec in specs)

        # 가격
        spans = price_info.find_elements(By.CSS_SELECTOR, 'li > div span') 
        price_spec = ''
        price = ''

        for span in spans:
            if ('위' in span.text): continue  # 순위 제외
            if ('원' in span.text): break    # 첫번째 금액만 받고 종료

            if ('B' in span.text):
                price_spec = span.text
            else:
                price = span.text

        danawa_prices.append({
            'index': ((page_num - 1) * 90) + index,
            'computer_name': computer_name,
            'computer_spec': computer_spec,
            'price_spec' : price_spec,
            'price' : price
        })  

In [ ]:
df_total = pd.DataFrame(danawa_prices)

df_total.info()

In [ ]:
# 가격비교예정으로 금액이 없음
df_total[df_total['price'].isna()]


In [ ]:
# 가격이 없은 행은 삭제
df_total = df_total.dropna()

In [ ]:
# 판다스 컬럼에 천단위 쉼표(,)는 제거후 형변환
df_total['price'] = df_total['price'].str.replace(',', '')

In [ ]:
df_total['price'] = pd.to_numeric(df_total['price'])

In [ ]:
df_total.info()

In [ ]:
df_total.to_csv('../data/danawa_게이밍PC.csv', encoding='utf-8', index=False)